# Segmentación de Lesiones en Retinopatía Diabética: U-Net con Transfer Learning vs. Segment Anything

En el laboratorio 2.5 vimos que una red pre-entrenada en ImageNet puede reutilizarse para un problema nuevo sin entrenarla desde cero: **transfer learning**. En este laboratorio llevamos esa idea a la **segmentación semántica** de imágenes médicas, siguiendo la clase 2.6.

Trabajaremos con el conjunto de datos **IDRiD** (*Indian Diabetic Retinopathy Image Dataset*), que contiene 81 fotografías de fondo de ojo con máscaras a nivel de pixel de cuatro lesiones de retinopatía diabética: microaneurismas (MA), hemorragias (HE), exudados duros (EX) y exudados blandos (SE).

**Objetivos**

1. Construir una **U-Net con encoder ResNet-34 pre-entrenado** en ImageNet (transfer learning).
2. Comparar dos funciones de pérdida para **exudados duros**: entropía cruzada binaria (BCE) frente a **Dice + Focal**.
3. Evaluar **Segment Anything (SAM)** sin ajuste, con prompts de caja y de clics derivados del ground truth.
4. Discutir: ¿cuántos clics necesita SAM para igualar al U-Net? ¿En qué lesiones falla y por qué?

> Se recomienda ejecutar en Google Colab con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*).

In [ ]:
%pip install -q transformers opencv-python scikit-learn tqdm

In [2]:
import os
import glob
import copy
import random
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights
from sklearn.metrics import average_precision_score

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

/Users/sergio/code/AI-MDS/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Computación en GPU

Al igual que en el laboratorio 2.5, definimos la variable `device` para mover modelos y tensores a la GPU si está disponible.

In [ ]:
if hasattr(torch, 'cuda') and torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print('Computando en device = {}'.format(device))

## Parámetros del experimento

Todo el laboratorio se controla desde esta celda. Comenzaremos con **exudados duros** (`EX`), la lesión más abundante en píxeles; luego podrá repetir el experimento con otras lesiones.

In [ ]:
LESION = 'EX'              # 'MA', 'HE', 'EX', 'SE' u 'OD' (disco óptico)
IMG_H, IMG_W = 704, 1056   # resolución de trabajo: múltiplos de 32, aspecto ≈ 4288×2848
CROP = 384                 # tamaño de los parches de entrenamiento
CROPS_PER_IMAGE = 8        # parches por imagen en cada época
BATCH_SIZE = 8
EPOCHS = 40
LR = 3e-4
PRETRAINED = True          # encoder ResNet-34 con pesos de ImageNet
USE_CLAHE = True           # ecualización local de contraste como preprocesamiento
N_VAL = 6                  # imágenes de entrenamiento reservadas para validación

# Prompts para SAM
MAX_COMPONENTS = 40        # máximo de lesiones (componentes conexas) con prompt por imagen
MIN_AREA = 4               # área mínima (pixeles) de una componente para recibir prompt

## Conjunto de Datos IDRiD

IDRiD se distribuye en [IEEE Dataport](https://ieee-dataport.org/open-access/indian-diabetic-retinopathy-image-dataset-idrid) (acceso abierto, requiere una cuenta gratuita). Descargue el archivo **`A. Segmentation.zip`** y déjelo en la carpeta `data/` o en su Google Drive.

Su estructura es la siguiente:

```
A. Segmentation/
├── 1. Original Images/
│   ├── a. Training Set/   IDRiD_01.jpg … IDRiD_54.jpg
│   └── b. Testing Set/    IDRiD_55.jpg … IDRiD_81.jpg
└── 2. All Segmentation Groundtruths/
    ├── a. Training Set/
    │   ├── 1. Microaneurysms/   IDRiD_01_MA.tif …
    │   ├── 2. Haemorrhages/     IDRiD_01_HE.tif …
    │   ├── 3. Hard Exudates/    IDRiD_01_EX.tif …
    │   ├── 4. Soft Exudates/    IDRiD_01_SE.tif …
    │   └── 5. Optic Disc/       IDRiD_01_OD.tif …
    └── b. Testing Set/  (misma estructura)
```

El código busca los archivos por nombre, así que funciona aunque la carpeta raíz tenga otro nombre. No todas las imágenes tienen todas las lesiones: si falta la máscara, la lesión está ausente y usamos una máscara vacía.

In [ ]:
DATA_DIR = 'data/IDRiD'
DATA_ZIP = 'data/A. Segmentation.zip'

# Si dejó el archivo en Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ZIP = '/content/drive/MyDrive/IDRiD/A. Segmentation.zip'

if not glob.glob(os.path.join(DATA_DIR, '**', 'IDRiD_*.jpg'), recursive=True):
    if os.path.exists(DATA_ZIP):
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(DATA_DIR)
    else:
        raise FileNotFoundError(
            f'No se encontró IDRiD en {DATA_DIR} ni el archivo {DATA_ZIP}. '
            'Descargue "A. Segmentation.zip" desde IEEE Dataport (ver instrucciones arriba).')

In [ ]:
def find_pairs(split, lesion=LESION):
    # Devuelve [(ruta_imagen, ruta_mascara o None)] del split 'Training' o 'Testing'.
    images = sorted(p for p in glob.glob(os.path.join(DATA_DIR, '**', 'IDRiD_*.jpg'), recursive=True)
                    if split.lower() in p.lower())
    masks = {os.path.basename(m).rsplit(f'_{lesion}', 1)[0]: m
             for m in glob.glob(os.path.join(DATA_DIR, '**', f'IDRiD_*_{lesion}.tif'), recursive=True)
             if split.lower() in m.lower()}
    return [(p, masks.get(os.path.splitext(os.path.basename(p))[0])) for p in images]

train_pairs = find_pairs('Training')
test_pairs = find_pairs('Testing')
print(f'Entrenamiento: {len(train_pairs)} imágenes, {sum(m is not None for _, m in train_pairs)} con máscara de {LESION}')
print(f'Prueba:        {len(test_pairs)} imágenes, {sum(m is not None for _, m in test_pairs)} con máscara de {LESION}')

Las imágenes originales miden 4288×2848 pixeles. Las llevamos a la resolución de trabajo. Para la máscara usamos interpolación por área y marcamos como lesión todo pixel con algo de cobertura: así una lesión diminuta no desaparece al reducir la imagen.

In [ ]:
def load_pair(img_path, mask_path, size=(IMG_W, IMG_H)):
    img = np.array(Image.open(img_path).convert('RGB'))
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    if mask_path is None:
        mask = np.zeros((size[1], size[0]), dtype=np.uint8)
    else:
        m = np.array(Image.open(mask_path))
        if m.ndim == 3:
            m = m.max(axis=2)
        m = cv2.resize((m > 0).astype(np.float32), size, interpolation=cv2.INTER_AREA)
        mask = (m > 0).astype(np.uint8)
    return img, mask

train_raw = [load_pair(i, m) for i, m in tqdm(train_pairs, desc='Entrenamiento')]
test_raw = [load_pair(i, m) for i, m in tqdm(test_pairs, desc='Prueba')]

In [ ]:
def overlay(img, mask, color=(255, 255, 0), alpha=0.6):
    out = img.copy()
    out[mask > 0] = (alpha * np.array(color) + (1 - alpha) * out[mask > 0]).astype(np.uint8)
    return out

fig, axs = plt.subplots(2, 3, figsize=(15, 7))
for k, (img, mask) in enumerate(train_raw[:3]):
    axs[0, k].imshow(img)
    axs[1, k].imshow(overlay(img, mask))
    axs[0, k].set_title(os.path.basename(train_pairs[k][0]))
    axs[1, k].set_title(f'Máscara de {LESION}')
for ax in axs.ravel():
    ax.axis('off')
plt.tight_layout()

### Desbalance de clases

Como vimos en clase, casi todos los pixeles son fondo. Esto vuelve engañosa la exactitud de pixel y motiva pérdidas como Dice o Focal.

In [ ]:
frac = np.mean([m.mean() for _, m in train_raw])
print(f'Fracción promedio de pixeles con {LESION}: {100 * frac:.2f}%')
print(f'Un modelo que predice "fondo" en todo tendría una exactitud de pixel de {100 * (1 - frac):.2f}%')

## Preprocesamiento

Dos transformaciones clásicas en fondo de ojo:

* **Canal verde:** concentra el contraste de las lesiones rojas (MA, HE) y amarillas (EX).
* **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*): ecualiza el contraste localmente. La aplicamos sobre la luminancia en el espacio LAB para conservar el color.

In [ ]:
def clahe_rgb(img, clip=2.0, tiles=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles))
    lab[..., 0] = clahe.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

img = train_raw[0][0]
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].imshow(img); axs[0].set_title('Original')
axs[1].imshow(img[..., 1], cmap='gray'); axs[1].set_title('Canal verde')
axs[2].imshow(clahe_rgb(img)); axs[2].set_title('CLAHE')
for ax in axs:
    ax.axis('off')

## Datasets y DataLoaders

Para el entrenamiento extraemos **parches aleatorios** de 384×384. Con probabilidad 0,7 el parche se centra en un pixel de lesión (sobremuestreo de la clase minoritaria) y aplicamos volteos como aumento de datos. La validación y la prueba se hacen sobre la **imagen completa**.

Normalizamos con la media y desviación de ImageNet, igual que `std_normalize` en el laboratorio 2.5, porque el encoder fue pre-entrenado con esa normalización.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def to_tensor(img):
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

prep = clahe_rgb if USE_CLAHE else (lambda x: x)
train_all = [(prep(img), mask) for img, mask in train_raw]
test_data = [(prep(img), mask) for img, mask in test_raw]

idx = list(range(len(train_all)))
random.Random(0).shuffle(idx)
val_data = [train_all[i] for i in idx[:N_VAL]]
train_data = [train_all[i] for i in idx[N_VAL:]]
print(f'Entrenamiento: {len(train_data)}, validación: {len(val_data)}, prueba: {len(test_data)}')

In [ ]:
class PatchDataset(torch.utils.data.Dataset):
    # Parches aleatorios; con probabilidad p_lesion el parche se centra en un pixel de lesión.
    def __init__(self, data, crop=CROP, n_per_image=CROPS_PER_IMAGE, p_lesion=0.7):
        self.data, self.crop, self.n, self.p = data, crop, n_per_image, p_lesion
        self.lesion_px = [np.argwhere(m > 0) for _, m in data]

    def __len__(self):
        return len(self.data) * self.n

    def __getitem__(self, idx):
        k = idx // self.n
        img, mask = self.data[k]
        H, W = mask.shape
        c = self.crop
        if len(self.lesion_px[k]) and random.random() < self.p:
            y, x = self.lesion_px[k][random.randrange(len(self.lesion_px[k]))]
            y0 = int(np.clip(y - c // 2, 0, H - c))
            x0 = int(np.clip(x - c // 2, 0, W - c))
        else:
            y0, x0 = random.randint(0, H - c), random.randint(0, W - c)
        im, m = img[y0:y0 + c, x0:x0 + c], mask[y0:y0 + c, x0:x0 + c]
        if random.random() < 0.5:
            im, m = im[:, ::-1], m[:, ::-1]
        if random.random() < 0.5:
            im, m = im[::-1], m[::-1]
        return to_tensor(np.ascontiguousarray(im)), torch.from_numpy(np.ascontiguousarray(m)).float().unsqueeze(0)

train_loader = torch.utils.data.DataLoader(PatchDataset(train_data), batch_size=BATCH_SIZE,
                                           shuffle=True, num_workers=2, drop_last=True)

In [ ]:
for images, masks in train_loader:
    print(images.shape, masks.shape)
    break

## Modelo: U-Net con Encoder Pre-entrenado

En el laboratorio 2.5 reutilizamos las capas convolucionales de VGG-16 como **extractor de características**. Aquí hacemos lo mismo con **ResNet-34** pre-entrenada en ImageNet, pero en vez de una capa de clasificación agregamos un **decodificador** que recupera la resolución espacial, con **conexiones de salto** desde cada nivel del encoder (la idea central de U-Net).

| Nivel del encoder | Resolución | Canales |
|---|---|---|
| `conv1` (stem) | 1/2 | 64 |
| `layer1` | 1/4 | 64 |
| `layer2` | 1/8 | 128 |
| `layer3` | 1/16 | 256 |
| `layer4` | 1/32 | 512 |

El encoder parte con pesos de ImageNet y el decodificador se entrena desde cero. Por eso la resolución de entrada debe ser múltiplo de 32.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNetResNet34(nn.Module):
    def __init__(self, n_classes=1, pretrained=True):
        super().__init__()
        r = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)   # 1/2,  64
        self.pool = r.maxpool                               # 1/4
        self.enc1, self.enc2 = r.layer1, r.layer2           # 1/4: 64, 1/8: 128
        self.enc3, self.enc4 = r.layer3, r.layer4           # 1/16: 256, 1/32: 512
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, n_classes, kernel_size=1)

    def encoder(self):
        return [self.stem, self.enc1, self.enc2, self.enc3, self.enc4]

    def forward(self, x):
        s0 = self.stem(x)
        e1 = self.enc1(self.pool(s0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d = self.dec4(e4, e3)
        d = self.dec3(d, e2)
        d = self.dec2(d, e1)
        d = self.dec1(d, s0)
        d = self.dec0(d)
        return self.head(d)   # logits, misma resolución que la entrada

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = UNetResNet34(pretrained=PRETRAINED)
print(f'El modelo tiene {count_parameters(model):,} parámetros entrenables')
print('Salida:', model(torch.zeros(1, 3, CROP, CROP)).shape)

Igual que en el laboratorio 2.5, podemos **congelar** el encoder y entrenar solo el decodificador. Esto reduce el cómputo, pero el encoder no se adapta al fondo de ojo; lo exploraremos en las preguntas finales.

In [ ]:
for block in model.encoder():
    for parameter in block.parameters():
        parameter.requires_grad = False
print(f'Con el encoder congelado: {count_parameters(model):,} parámetros entrenables')
del model

## Funciones de Pérdida

* **BCE:** $\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_i \left[y_i \log p_i + (1-y_i)\log(1-p_i)\right]$. La clase mayoritaria domina el gradiente.
* **Focal:** $\mathcal{L}_{\text{FL}} = -\alpha_t (1-p_t)^\gamma \log p_t$. Reduce el peso de los pixeles fáciles.
* **Dice:** $\mathcal{L}_{\text{Dice}} = 1 - \frac{2\sum_i p_i g_i + \varepsilon}{\sum_i p_i + \sum_i g_i + \varepsilon}$. Optimiza directamente el solapamiento.

Combinamos Focal y Dice con pesos 20:1, la misma proporción que usa SAM 2 en su entrenamiento.

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def focal_loss(logits, target, alpha=0.25, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    p_t = torch.exp(-bce)
    alpha_t = alpha * target + (1 - alpha) * (1 - target)
    return (alpha_t * (1 - p_t) ** gamma * bce).mean()

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    num = 2 * (p * target).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return (1 - num / den).mean()

def dice_focal_loss(logits, target, w_focal=20.0, w_dice=1.0):
    return w_focal * focal_loss(logits, target) + w_dice * dice_loss(logits, target)

## Métricas

Reportamos **Dice** e **IoU** agregados sobre todos los pixeles del conjunto, el Dice promedio por imagen y el **AUPR** (área bajo la curva precisión-exhaustividad), la métrica oficial del desafío IDRiD. El AUPR no depende del umbral de decisión.

In [ ]:
def dice_iou(pred, target):
    inter = np.logical_and(pred, target).sum()
    ps, ts = pred.sum(), target.sum()
    dice = 2 * inter / (ps + ts) if ps + ts > 0 else 1.0
    iou = inter / (ps + ts - inter) if ps + ts > 0 else 1.0
    return dice, iou

def summarize(preds, data, probs=None, stride=4):
    # preds: lista de máscaras binarias; probs: lista de mapas de probabilidad (para AUPR).
    inter = ps = ts = 0
    per_image = []
    for pred, (_, mask) in zip(preds, data):
        gt = mask > 0
        inter += np.logical_and(pred, gt).sum(); ps += pred.sum(); ts += gt.sum()
        per_image.append(dice_iou(pred, gt)[0])
    res = {'Dice': 2 * inter / (ps + ts), 'IoU': inter / (ps + ts - inter),
           'Dice por imagen': float(np.mean(per_image))}
    if probs is not None:
        y = np.concatenate([m[::stride, ::stride].ravel() for _, m in data])
        s = np.concatenate([p[::stride, ::stride].ravel() for p in probs])
        res['AUPR'] = average_precision_score(y, s) if y.any() else float('nan')
    return res

@torch.no_grad()
def predict_proba(model, img):
    model.eval()
    x = to_tensor(img).unsqueeze(0).to(device)
    return torch.sigmoid(model(x))[0, 0].float().cpu().numpy()

def evaluate(model, data, thr=0.5):
    probs = [predict_proba(model, img) for img, _ in data]
    return summarize([p > thr for p in probs], data, probs)

## Entrenamiento

El ciclo de entrenamiento sigue el del laboratorio 2.5 y guarda un `history`. Usamos AdamW con tasa de aprendizaje coseno y precisión mixta en GPU. Al final conservamos los pesos de la época con mejor Dice de validación.

In [ ]:
def train_model(loss_fn, name, epochs=EPOCHS, lr=LR, freeze_encoder=False):
    model = UNetResNet34(pretrained=PRETRAINED).to(device)
    if freeze_encoder:
        for block in model.encoder():
            for parameter in block.parameters():
                parameter.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    use_amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp) if use_amp else None
    history = {'train_loss': [], 'val_dice': [], 'val_aupr': []}
    best_dice, best_state = -1.0, None

    for epoch in range(epochs):
        model.train()
        running_loss, total = 0.0, 0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                    logits = model(images)
            else:
                logits = model(images)
            loss = loss_fn(logits.float(), masks)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            running_loss += loss.item() * images.size(0)
            total += images.size(0)
        scheduler.step()

        val = evaluate(model, val_data)
        history['train_loss'].append(running_loss / total)
        history['val_dice'].append(val['Dice'])
        history['val_aupr'].append(val['AUPR'])
        if val['Dice'] > best_dice:
            best_dice, best_state = val['Dice'], copy.deepcopy(model.state_dict())
        print(f'[{name}] Época {epoch + 1}/{epochs} - loss: {running_loss / total:.4f} '
              f'- val Dice: {val["Dice"]:.4f} - val AUPR: {val["AUPR"]:.4f}')

    model.load_state_dict(best_state)
    return model, history

### Experimento A: entropía cruzada binaria

In [ ]:
model_bce, history_bce = train_model(bce_loss, 'BCE')

### Experimento B: Dice + Focal

In [ ]:
model_df, history_df = train_model(dice_focal_loss, 'Dice+Focal')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for h, label in [(history_bce, 'BCE'), (history_df, 'Dice + Focal')]:
    axs[0].plot(h['train_loss'], label=label)
    axs[1].plot(h['val_dice'], label=label)
    axs[2].plot(h['val_aupr'], label=label)
for ax, title in zip(axs, ['Pérdida de entrenamiento', 'Dice de validación', 'AUPR de validación']):
    ax.set_title(title); ax.set_xlabel('Época'); ax.legend()
axs[0].set_yscale('log')
plt.tight_layout()

### Evaluación en el conjunto de prueba

Evaluamos ambos modelos en las 27 imágenes de prueba de IDRiD.

In [ ]:
results = {
    'U-Net BCE': evaluate(model_bce, test_data),
    'U-Net Dice+Focal': evaluate(model_df, test_data),
}
pd.DataFrame(results).T.round(4)

In [ ]:
def error_map(img, pred, gt):
    # Verdaderos positivos en amarillo, falsos positivos en magenta y falsos negativos en cian.
    out = img.copy()
    out[pred & gt] = (255, 220, 0)
    out[pred & ~gt] = (255, 0, 255)
    out[~pred & gt] = (0, 220, 255)
    return out

fig, axs = plt.subplots(3, 3, figsize=(16, 11))
for r, (img, mask) in enumerate(test_raw[:3]):
    x, gt = test_data[r][0], mask > 0
    axs[r, 0].imshow(overlay(img, mask)); axs[r, 0].set_title('Ground truth')
    axs[r, 1].imshow(error_map(img, predict_proba(model_bce, x) > 0.5, gt)); axs[r, 1].set_title('U-Net BCE')
    axs[r, 2].imshow(error_map(img, predict_proba(model_df, x) > 0.5, gt)); axs[r, 2].set_title('U-Net Dice + Focal')
for ax in axs.ravel():
    ax.axis('off')
plt.tight_layout()

## Segment Anything (SAM)

SAM (Kirillov et al., 2023) separa el trabajo en dos partes: un **codificador de imagen** pesado (ViT), que se ejecuta **una vez por imagen**, y un **decodificador de máscaras** liviano, que responde a cada *prompt* (puntos, cajas o máscaras). SAM no conoce las clases: devuelve la máscara del objeto señalado.

Para evaluarlo **simulamos** los prompts desde el ground truth, siguiendo el protocolo de los papers de SAM y SAM 2:

* **Caja:** el rectángulo que encierra cada lesión (componente conexa).
* **Clics:** el primer clic positivo va en el centro de la lesión (el pixel más alejado de su borde). Cada clic siguiente va en el centro de la mayor región de error: positivo si es un falso negativo, negativo si es un falso positivo.

Ojo: estos prompts son un **oráculo**, porque usan el ground truth. En la práctica los entregaría un oftalmólogo.

Usamos la implementación de SAM de Hugging Face `transformers`. El modelo `vit-base` cabe cómodamente en una GPU T4.

In [ ]:
from transformers import SamModel, SamProcessor

SAM_CHECKPOINT = 'facebook/sam-vit-base'   # alternativas: 'facebook/sam-vit-large', 'facebook/sam-vit-huge'
sam_model = SamModel.from_pretrained(SAM_CHECKPOINT).to(device).eval()
sam_processor = SamProcessor.from_pretrained(SAM_CHECKPOINT)
print(f'SAM tiene {count_parameters(sam_model):,} parámetros')

In [ ]:
def lesion_components(mask, max_components=MAX_COMPONENTS, min_area=MIN_AREA):
    # Componentes conexas de la máscara, de mayor a menor área.
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    comps = sorted(((stats[i, cv2.CC_STAT_AREA], i) for i in range(1, n)
                    if stats[i, cv2.CC_STAT_AREA] >= min_area), reverse=True)[:max_components]
    return [labels == i for _, i in comps]

def box_from_mask(region, margin=2):
    ys, xs = np.where(region)
    return [float(xs.min() - margin), float(ys.min() - margin),
            float(xs.max() + margin), float(ys.max() + margin)]

def center_point(region):
    # Pixel de la región más alejado de su borde (protocolo de SAM / SAM 2).
    dist = cv2.distanceTransform(np.pad(region.astype(np.uint8), 1), cv2.DIST_L2, 5)[1:-1, 1:-1]
    y, x = np.unravel_index(dist.argmax(), dist.shape)
    return [float(x), float(y)]

El embedding de la imagen se calcula una sola vez y se reutiliza para todos los prompts: la idea de *codificar una vez, decodificar muchas* que vimos en clase.

In [ ]:
@torch.no_grad()
def sam_embed(img):
    inputs = sam_processor(images=img, return_tensors='pt').to(device)
    return sam_model.get_image_embeddings(inputs['pixel_values'])

@torch.no_grad()
def sam_predict(img, embedding, boxes=None, points=None, labels=None):
    # Una máscara por prompt. boxes: [[x0, y0, x1, y1], ...]; points: [[[x, y], ...], ...]
    kwargs = {}
    if boxes is not None:
        kwargs['input_boxes'] = [boxes]
    if points is not None:
        kwargs['input_points'] = [points]
        kwargs['input_labels'] = [labels]
    inputs = sam_processor(images=img, return_tensors='pt', **kwargs).to(device)
    inputs.pop('pixel_values')
    outputs = sam_model(image_embeddings=embedding, multimask_output=False, **inputs)
    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs['original_sizes'].cpu(), inputs['reshaped_input_sizes'].cpu())[0]
    return masks[:, 0].numpy().astype(bool)   # (n_prompts, H, W)

def sam_segment(img, mask, embedding, mode='box', n_clicks=1):
    comps = lesion_components(mask)
    if not comps:
        return np.zeros(mask.shape, dtype=bool)
    if mode == 'box':
        masks = sam_predict(img, embedding, boxes=[box_from_mask(c) for c in comps])
    else:
        points = [[center_point(c)] for c in comps]
        labels = [[1] for _ in comps]
        masks = sam_predict(img, embedding, points=points, labels=labels)
        for _ in range(n_clicks - 1):
            for i, c in enumerate(comps):
                fn, fp = c & ~masks[i], masks[i] & ~c
                if fn.any() and fn.sum() >= fp.sum():
                    points[i].append(center_point(fn)); labels[i].append(1)
                elif fp.any():
                    points[i].append(center_point(fp)); labels[i].append(0)
                else:
                    points[i].append(points[i][-1]); labels[i].append(labels[i][-1])
            masks = sam_predict(img, embedding, points=points, labels=labels)
    return masks.any(axis=0)

Ahora evaluamos SAM en el conjunto de prueba con cuatro tipos de prompt. Usamos la imagen **original** (sin CLAHE), porque SAM fue entrenado con fotografías naturales.

In [ ]:
SAM_MODES = {'SAM caja': ('box', 1), 'SAM 1 clic': ('click', 1),
             'SAM 3 clics': ('click', 3), 'SAM 5 clics': ('click', 5)}
sam_preds = {name: [] for name in SAM_MODES}

for img, mask in tqdm(test_raw, desc='SAM'):
    embedding = sam_embed(img)
    for name, (mode, k) in SAM_MODES.items():
        sam_preds[name].append(sam_segment(img, mask, embedding, mode=mode, n_clicks=k))

for name, preds in sam_preds.items():
    results[name] = summarize(preds, test_raw)

results_df = pd.DataFrame(results).T
results_df.round(4)

In [ ]:
ax = results_df['Dice'].plot.barh(figsize=(8, 4), color=['#144186', '#144186'] + ['#04ABE2'] * len(SAM_MODES))
ax.set_xlabel(f'Dice en prueba ({LESION})')
ax.invert_yaxis()
for i, v in enumerate(results_df['Dice']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
plt.tight_layout()

Comparemos visualmente en una zona ampliada de una imagen de prueba. Las lesiones pequeñas son justamente donde aparecen las diferencias.

In [ ]:
k = 0
img, mask = test_raw[k]
gt = mask > 0
ys, xs = np.where(gt)
cy, cx = (int(np.median(ys)), int(np.median(xs))) if len(ys) else (IMG_H // 2, IMG_W // 2)
y0, x0 = np.clip(cy - 150, 0, IMG_H - 300), np.clip(cx - 200, 0, IMG_W - 400)
zoom = (slice(y0, y0 + 300), slice(x0, x0 + 400))

panels = [('Ground truth', overlay(img, mask)),
          ('U-Net Dice + Focal', error_map(img, predict_proba(model_df, test_data[k][0]) > 0.5, gt)),
          ('SAM caja', error_map(img, sam_preds['SAM caja'][k], gt)),
          ('SAM 3 clics', error_map(img, sam_preds['SAM 3 clics'][k], gt))]
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (title, im) in zip(axs, panels):
    ax.imshow(im[zoom]); ax.set_title(title); ax.axis('off')
plt.tight_layout()

## MedSAM (opcional)

MedSAM (Ma et al., *Nature Communications*, 2024) es SAM ajustado con más de 1,5 millones de pares imagen-máscara médicas. Fue entrenado con prompts de **caja**, así que lo evaluamos solo con ese modo. Cambie `RUN_MEDSAM = True` para ejecutarlo.

In [ ]:
RUN_MEDSAM = False

if RUN_MEDSAM:
    sam_model = SamModel.from_pretrained('flaviagiammarino/medsam-vit-base').to(device).eval()
    sam_processor = SamProcessor.from_pretrained('flaviagiammarino/medsam-vit-base')
    preds = [sam_segment(img, mask, sam_embed(img), mode='box') for img, mask in tqdm(test_raw, desc='MedSAM')]
    results['MedSAM caja'] = summarize(preds, test_raw)
    results_df = pd.DataFrame(results).T
    display(results_df.round(4))

## Preguntas para el Informe

1. **Umbral y AUPR.** ¿Por qué el umbral 0,5 perjudica al modelo entrenado con BCE? Compare su Dice con su AUPR y busque el umbral que maximiza el Dice en validación.
2. **Transfer learning.** Entrene con `train_model(dice_focal_loss, 'congelado', freeze_encoder=True)`, como en el laboratorio 2.5. ¿Cuántos parámetros entrenables quedan y cuánto cae el Dice? Repita con `PRETRAINED = False`.
3. **Otras lesiones.** Repita todo con `LESION = 'MA'` y luego `'SE'`. ¿Qué ocurre con los microaneurismas al reducir la resolución? Pruebe con `IMG_H, IMG_W = 1408, 2112`.
4. **SAM.** ¿Cuántos clics necesita SAM para igualar al U-Net? ¿Es justa la comparación, considerando que los prompts vienen del ground truth y que solo se marcan las `MAX_COMPONENTS` lesiones más grandes?
5. **Preprocesamiento para SAM.** ¿Mejora SAM si recibe la imagen con CLAHE? ¿Por qué podría empeorar?

**Entregable:** este notebook ejecutado y un informe breve con una tabla de métricas por lesión y por método.

## Referencias

* Porwal, P. et al. (2018). Indian Diabetic Retinopathy Image Dataset (IDRiD). *Data*, 3(3), 25.
* Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation. *MICCAI*.
* He, K. et al. (2016). Deep Residual Learning for Image Recognition. *CVPR*.
* Lin, T.-Y. et al. (2017). Focal Loss for Dense Object Detection. *ICCV*.
* Milletari, F., Navab, N., & Ahmadi, S.-A. (2016). V-Net. *3DV*.
* Kirillov, A. et al. (2023). Segment Anything. *ICCV*.
* Ravi, N. et al. (2025). SAM 2: Segment Anything in Images and Videos. *ICLR*.
* Ma, J. et al. (2024). Segment Anything in Medical Images. *Nature Communications*, 15, 654.